# PRESENTACIÓN DE VARIABLES FEATURE ENGINEERING
A continuación realizaremos el feature engineering del dataframe para la primera competencia. La idea es crear variables interesantes que nos ayuden a la hora de clusterizar mejor a los individuos de todo el set de datos, pero, en particular, los baja+2 que debemos identificar con patrones para la presentación audiovisual del día 27/9

El primer paso será la creación de la clase ternaria según los lineamientos de clase.
El segundo paso la creación, sobre este nuevo dataframe, de las columnas creadas en el diccionario.
Luego haremos una pequeña explicación de cada nuevo atributo (poseerá un ID que es meramente identificatorio, ignorarlo)

## PRIMER PASO: Generación de clase ternaria

In [2]:
require("data.table")

PARAM <- list()
PARAM$data_dir <- "C:/Users/germa/Desktop/Personal/Maestría/Data Mining/Primer Año/Data Mining en Economía y Finanzas/Clase1"

archivo_crudo    <- file.path(PARAM$data_dir, "competencia_01_crudo.csv")
archivo_ternaria <- file.path(PARAM$data_dir, "competencia_01.csv.gz")

if (!file.exists(archivo_crudo)) {
  stop("No encuentro el archivo crudo en: ", archivo_crudo)
}

if (file.exists(archivo_ternaria)) {
  # ya esta generada de una corrida anterior, la leo directo
  dataset <- fread(archivo_ternaria)
} else {
  # la genero desde el crudo
  dataset <- fread(archivo_crudo)

  dsimple <- dataset[, list(
      "pos" = .I,
      numero_de_cliente,
      periodo0 = as.integer(foto_mes/100)*12 + foto_mes%%100 ) ]

  setorder(dsimple, numero_de_cliente, periodo0)

  periodo_ultimo <- dsimple[, max(periodo0)]
  periodo_anteultimo <- periodo_ultimo - 1

  dsimple[, c("periodo1", "periodo2") :=
      shift(periodo0, n = 1:2, fill = NA, type = "lead"), numero_de_cliente]

  dsimple[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]

  dsimple[periodo0 < periodo_ultimo &
      (is.na(periodo1) | periodo0 + 1 < periodo1),
      clase_ternaria := "BAJA+1"]

  dsimple[periodo0 < periodo_anteultimo & (periodo0 + 1 == periodo1) &
      (is.na(periodo2) | periodo0 + 2 < periodo2),
      clase_ternaria := "BAJA+2"]

  setorder(dsimple, pos)
  dataset[, clase_ternaria := dsimple$clase_ternaria]

  fwrite(dataset, file = archivo_ternaria, sep = ",")
}

# chequeo rapido
dataset[, .N, list(foto_mes, clase_ternaria)]


Cargando paquete requerido: data.table



foto_mes,clase_ternaria,N
<int>,<chr>,<int>
202103,CONTINUA,160921
202103,BAJA+2,960
202103,BAJA+1,1019
202104,CONTINUA,161181
202104,BAJA+2,1139
202104,BAJA+1,964
202105,CONTINUA,161755
202105,BAJA+1,1143
202105,BAJA+2,870


## SEGUNDO PASO: Feature Engineering
La creación de columnas es sencilla... lleva esta expresión

*dataset[, nombre_columna_nueva := expresion]*

In [5]:
div_segura <- function(num, denom) ifelse(denom == 0 | is.na(denom), NA_real_, num / denom)

# Id 1001: mide el consumo total de tarjetas
dataset[, consumo_tarjetas := mtarjeta_visa_consumo +mtarjeta_master_consumo]

# Id 1002: mide el saldo total de tarjetas
dataset[, saldo_tarjetas := Master_msaldopesos + Master_msaldodolares + Visa_msaldopesos + Visa_msaldodolares]

# Id 1028: mide los límites de compra. Es una variable importante para el modelo; calculo que debe medir que tanto dinero maneja un cliente -que tan importante es en tal caso-
dataset[, limites_compra := Visa_mlimitecompra + Master_mlimitecompra]

# Id 1003: mide cuando puede el cliente apalancarse en tarjetas sobre el limite de compra
dataset[, apalancamiento_tc := div_segura(saldo_tarjetas, limites_compra)]

# Id 1004: mide el endeudamiento total a largo plazo
dataset[, endeudamiento := mprestamos_personales + mprestamos_prendarios + mprestamos_hipotecarios]

# Id 1005: mide los activos del clientes que son los pasivos del banco. Ojo que esto da una rentabilidad que tiene un calculo propio.
dataset[, pasivos_banca_inversiones := mforex_buy - mforex_sell + mplazo_fijo_dolares + mplazo_fijo_pesos + minversion1_pesos + minversion1_dolares + minversion2]

# Id 1006: mide los pasivos totales del banco compuestos por los activos + el saldo de cuenta del cliente.
dataset[, pasivos_total_banco := pasivos_banca_inversiones + mcuentas_saldo]

# Id 1007: mide la cantidad de seguros que tiene el cliente
dataset[, seguros_cantidad := cseguro_vida + cseguro_auto + cseguro_vivienda]

# Id 1008: mide el ingreso total del cliente cuando este trabaja en realcion de dependencia.
dataset[, payroll := mpayroll + mpayroll2]

# Id 1009: mide el nivel de digitalización del cliente, es decir, que tanto usa canales digitales para operar con el banco
dataset[, nivel_digital := ccuenta_debitos_automaticos + ctarjeta_visa_debitos_automaticos + ctarjeta_master_debitos_automaticos + cpagodeservicios + cpagomiscuentas + chomebanking_transacciones + cmobile_app_trx]

# Id 1010: mide la cantidad de descuentos que tuvo el cliente. Se mide en cantidades por que muestra uso de tarjetas.
dataset[, descuentos_tarjeta := ccajeros_propios_descuentos + ctarjeta_visa_descuentos + ctarjeta_master_descuentos]

# Id 1011: mide el monto de las comisiones del banco. 
dataset[, comisiones := mcomisiones_mantenimiento + mcomisiones_otras]

# Id 1012: mide el nivel de analogía del cliente, es decir, que tanto usa canales físicos para operar con el banco
dataset[, nivel_analogico := ccajas_transacciones +ccajas_consultas + ccajas_depositos + ccajas_extracciones + ccajas_otras + catm_trx + catm_trx_other + cextraccion_autoservicio + ccallcenter_transacciones]

# Id 1013: mide el flujo neto de transferencias del cliente, es decir, que tanto dinero entra y sale de su cuenta
dataset[, transferencias_netas := mtransferencias_recibidas - mtransferencias_emitidas]

# Id 1014: mide el flujo neto de cheques del cliente, es decir, que tanto dinero entra y sale de su cuenta
dataset[, cheques_netos := mcheques_depositados - mcheques_emitidos]

# Id 1015: mide el flujo de endeudamiento por cheque y consumo que tiene el cliente. Es lo más parecido al flujo dentro del dataset
dataset[, endeudamiento_flujo :=mcheques_emitidos_rechazados - mcheques_depositados_rechazados + consumo_tarjetas]

# Id 1016: mide el ingreso neto del cliente, es decir, que tanto dinero entra a su cuenta
dataset[, ingresos_netos := payroll + cheques_netos + transferencias_netas]

# Id 1017: mide la capacidad de pago del cliente sobre el endeudamiento de tarjetas de crédito
dataset[, capacidad_pago_TC := div_segura(ingresos_netos, saldo_tarjetas)]

# Id 1018: mide la capacidad de pago del cliente TOTAL sobre el endeudamiento de préstamos. Ojo que son dos cosas distintas; buscamos entender si puede pagar una cuota - es obvio que el prestamo no podra pagarlo con activos por que siempre sera mayor-
dataset[, capacidad_pago_LP := div_segura(pasivos_total_banco, endeudamiento)]

# Id 1019: mide cuanto queda de saldo en las cuentas en relación a sus ingresos.
dataset[, saldo_sobre_ingresos := div_segura(mcuentas_saldo, ingresos_netos)]

# Id 1020: mide el pago de cuotas de préstamos sobre los ingresos netos.
dataset[, pago_cuotas_lp := div_segura(ingresos_netos, endeudamiento)]

# Id 1031: mide la capacidad de pago del endeudamiento de flujo.
dataset[, pago_endeudamiento_flujo := div_segura(mcuentas_saldo, endeudamiento_flujo)]

# Id 1021: mide la capacidad de pago del cliente TOTAL sobre el endeudamiento TOTAL.
dataset[, capacidad_pago_total := div_segura(pasivos_total_banco, endeudamiento + saldo_tarjetas + endeudamiento_flujo)]

# Id 1022: mide la rentabilidad de los activos del banco sobre la rentabilidad total
dataset[, share_rentabilidad_activos := div_segura(mactivos_margen, mrentabilidad)]

# Id 1023: mide la rentabilidad de los pasivos del banco sobre la rentabilidad total
dataset[, share_rentabilidad_pasivos := div_segura(mpasivos_margen, mrentabilidad)]

# Id 1024: se calcula cual es la rentabilidad que no proviene de intereses y dinero depositado.
dataset[, otra_rentabilidad := mrentabilidad - mpasivos_margen - mactivos_margen]

# Id 1025: mide la otra_rentabilidad sobre los ingresos netos (que no van a inversiones) 
dataset[, rentabilidad_ingresos := div_segura(otra_rentabilidad, ingresos_netos)]

# Id 1026: mide la rentabilidad total de cada familia de productos del cliente.
dataset[, rentabilidad_por_producto := div_segura(mrentabilidad, cproductos)]

# Id 1030: mide la relación entre el nivel de digitalización y el nivel de analogía del cliente. 
dataset[, digital_analogico := div_segura(nivel_digital, nivel_analogico)]

# Id 1027 — rentabilidad de cada fila sobre la mediana global
mediana_rentabilidad_global <- dataset[, median(mrentabilidad, na.rm = TRUE)]
dataset[, rentabilidad_vs_mediana_global := div_segura(mrentabilidad, mediana_rentabilidad_global)]

# Id 1032 — mediana de rentabilidad POR clase_ternaria, y la fila sobre la mediana de SU clase
dataset[, mediana_rentabilidad_clase := median(mrentabilidad, na.rm = TRUE), by = clase_ternaria]
dataset[, rentabilidad_vs_mediana_clase := div_segura(mrentabilidad, mediana_rentabilidad_clase)]

# Id 1029 — limites_compra de cada fila sobre la mediana global
mediana_limites_global <- dataset[, median(limites_compra, na.rm = TRUE)]
dataset[, limites_vs_mediana_global := div_segura(limites_compra, mediana_limites_global)]

# Id 1033 — mediana de limites_compra POR clase_ternaria, y la fila sobre la mediana de SU clase
dataset[, mediana_limites_clase := median(limites_compra, na.rm = TRUE), by = clase_ternaria]
dataset[, limites_vs_mediana_clase := div_segura(limites_compra, mediana_limites_clase)]

# Id 1034: flag que indica si la rentabilidad total del cliente (mrentabilidad) fue negativa.
# Acompaña a los ratios que dividen por mrentabilidad (share_rentabilidad_activos, share_rentabilidad_pasivos,
# rentabilidad_por_producto, rentabilidad_vs_mediana_global, rentabilidad_vs_mediana_clase), para que el modelo
# pueda distinguir los casos donde el signo de esos ratios no es directamente interpretable.
dataset[, flag_mrentabilidad_negativa := as.integer(mrentabilidad < 0)]


In [6]:
ncol(dataset)  # deberia haber crecido en 33 respecto al dataset original
summary(dataset[, .(consumo_tarjetas, endeudamiento, rentabilidad_vs_mediana_clase, digital_analogico)])

[1] 191

 consumo_tarjetas  endeudamiento      rentabilidad_vs_mediana_clase
 Min.   :      0   Min.   :       0   Min.   :-195.05900           
 1st Qu.:   7565   1st Qu.:       0   1st Qu.:   0.05106           
 Median :  24444   Median :       0   Median :   1.00000           
 Mean   :  43236   Mean   :   63910   Mean   :   1.73526           
 3rd Qu.:  53709   3rd Qu.:       0   3rd Qu.:   2.34526           
 Max.   :9410855   Max.   :41268290   Max.   : 556.64845           
                                                                   
 digital_analogico
 Min.   :   0.00  
 1st Qu.:   1.30  
 Median :   5.00  
 Mean   :  14.25  
 3rd Qu.:  15.33  
 Max.   :4792.50  
 NA's   :410761   

Cabe destacar que aquellos atributos relacionales (ratios) pueden tomar valores entre 0 y 1, y mayores a 1 cuando el numerador supera al denominador. Por ejemplo, casos como una persona mucho más digitalizada que analógica. Caso contrario, quien casi no interactúa por canales digitales tendrá valores cercanos a 0.

A su vez, es importante mencionar que existe la posibilidad de tener valores negativos. Esto puede darse de dos formas distintas: por un lado, cuando el numerador es negativo (por ejemplo, mrentabilidad, mactivos_margen o mpasivos_margen pueden tomar valores negativos), el ratio resultante también lo será. Por otro lado, cuando es el denominador el que puede cambiar de signo — como pasa en los ratios que dividen por mrentabilidad (share_rentabilidad_activos, share_rentabilidad_pasivos, rentabilidad_por_producto, los _vs_mediana_*) — el signo del ratio no siempre refleja de forma directa lo que pasó con el numerador: un mactivos_margen positivo dividido por un mrentabilidad negativo da como resultado un ratio negativo, aunque la contribución de activos haya sido positiva. Por eso, un ratio negativo no debe leerse automáticamente como "esa magnitud fue negativa": puede ser el numerador, el denominador, o ambos (en cuyo caso el ratio vuelve a ser positivo).

Finalmente, cabe aclarar que en los casos donde el denominador es exactamente 0 (por ejemplo, endeudamiento = 0, situación común en clientes sin préstamos), el ratio se define como NA en lugar de 0 o infinito, para evitar resultados artificiales. Esto explica la proporción de valores faltantes que se observa en columnas como capacidad_pago_LP o pago_cuotas_lp.

Como complemento a lo anterior, se agregó el flag `flag_mrentabilidad_negativa`, que indica si la rentabilidad total del cliente (`mrentabilidad`) fue negativa en ese período.

Este flag no reemplaza a los ratios, sino que los acompaña: `share_rentabilidad_activos`, `share_rentabilidad_pasivos`, `rentabilidad_por_producto`, `rentabilidad_vs_mediana_global` y `rentabilidad_vs_mediana_clase` dividen todos por `mrentabilidad`, así que cuando este denominador es negativo, el signo del ratio deja de ser directamente interpretable (puede invertirse si el numerador es positivo, o **volverse positivo "por casualidad" si numerador y denominador son negativos a la vez**). El flag le da al modelo una señal explícita de en qué régimen está parado, para que pueda distinguir estos casos sin depender únicamente del signo del ratio.

